# SageMaker AI MLflow with Strands Agents and Amazon Bedrock AgentCore

This tutorial we will introduce Strands Agents and how to deploy it through Amazon Bedrock AgentCore, we will also show you how to use MLflow for observability.

Install required libraries. You might get some dependency conflict errors but it shouldn't affect the functionality of the rest of this notebook.

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
!pip install -U sagemaker==2.253.1 datasets==4.4.1 mlflow==3.9.0 fsspec==2023.9.2 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

In [ ]:
# Install dependencies for AgentCore deployment. Ignore any warnings and residual dependency errors.
!pip install --force-reinstall -U -r requirements-agentcore.txt --quiet

In [ ]:
# Import deployment tools and utilities
import uuid
from utils import setup_cognito_user_pool, reauthenticate_user, delete_cognito_user_pool
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
from typing import Any, Optional
import urllib.parse
import requests
import json
import sagemaker

# Create a SageMaker session
sess = sagemaker.Session()

# Get the current AWS region
region = sess.boto_region_name

print(f"The current AWS region is: {region}")

## What happens behind the scenes?

To deploy our agents to `AgentCore Runtime`. To do so we need to:

- Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
- Initialize the App in our code with `app = BedrockAgentCoreApp()`
- Decorate the invocation function with the `@app.entrypoint` decorator
- Let AgentCore Runtime control the running of the agent with `app.run()`

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to AWS standards



> In this workshop the IAM permissioning needed for AgentCore runtime deployments are already configured for you. You can learn about the AgentCore runtime IAM permission settings in [AWS documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html).

### Setting up Amazon Cognito for Authentication

AgentCore Runtime requires authentication. We'll use Amazon Cognito to provide JWT tokens for accessing our deployed agent server.

In [ ]:
# Set up Amazon Cognito for AgentCore Runtime authentication
print("Setting up Amazon Cognito user pool...")

cognito_config = setup_cognito_user_pool()

print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

# Configure JWT authorization for AgentCore Runtime
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

### Deploying the agent to AgentCore Runtime with SageMaker AI MLflow

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables, and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** From a Operations/DevOps best practice is to package code as a container and push to ECR using CI/CD pipelines.

In this tutorial, we will use the Amazon Bedrock AgentCore Python SDK to package your artifacts and deploy them to AgentCore Runtime.

We will retrieve the stored notebook values containing the SageMaker AI MLflow App ARN. If the stored value is empty you can copy the MLflow App Tracking URI from the SageMaker AI MLflow studio console. You will enter the MLflow App Tracking URI for the variable `MLFLOW_TRACKING_URI`.

In [ ]:
# Retrieve values stored from previous labs
%store -r

%store
if MLFLOW_TRACKING_URI is None or MLFLOW_TRACKING_URI == '':
    print("MLFLOW_TRACKING_URI not found. Please enter your MLflow App ARN.")
MLFLOW_TRACKING_URI

### Configure AgentCore Runtime deployment with SageMaker AI MLflow
We will use `mlflow` python library to enable tracing for the strands agent sdk deployed in AgentCore. To enable tracing we call the `mlflow.autolog()` module which has the integrations needed to automatically process strands agent sdk framework and log the traces into SageMaker AI MLflow. Note: Strands agent framework is support in MLflow version `3.4.0` or higher.

In this workshop lab we will use Amazon Bedrock AgentCore Runtime [starter toolkit](https://github.com/aws/bedrock-agentcore-starter-toolkit) to simplyfy the AgentCore Runtime deployment with an entrypoint. The starter kit will configure the Dockerfile and auto-create the Amazon ECR repository on launch. The starter kit will consume the requirements file containing reference to MLflow `v3.4.0+` to generate the strands agent application code.

During the configure step, your Dockerfile will be generated based on your application code.

![runtime](./static/sagemaker-mlflow-agentCore.png)

Important: In the next notebook cell make sure to set the following variables, 
- `MLFLOW_TRACKING_URI`: Automatically retrieved from previous labs via `%store -r`
- `_SAGEMAKER_MLFLOW_EXPERIMENT_NAME`: A name for this MLflow experiment, set to `medical-triage-agentcore`
- `_REGION`: The aws region of the SageMaker AI MLflow App and the AgentCore agent, for example `us-east-1`
- `_BEDROCK_MODELID`: The Amazon Bedrock model ID, for example `global.anthropic.claude-haiku-4-5-20251001-v1:0`

For the agent use-case we will deploy the same **Medical Triage Agent** used in the previous labs to AgentCore Runtime, using [Strands Agents](https://strandsagents.com/latest/) with Amazon Bedrock as the model provider.

Medical Triage Agent: Assists healthcare staff by analyzing patient symptoms and recommending treatment protocols. It consists of 2 tools:
- `symptom_lookup`: Look up patient symptoms by patient ID
- `treatment_lookup`: Retrieve treatment protocol based on patient symptoms

<div class="alert alert-block alert-info">
<b>Note:</b> The MLflow Tracking URI is automatically retrieved from previous labs via <code>%store -r</code>.
</div>

In [ ]:
%%writefile strands_agentcore_SageMaker_deploy.py
# Create AgentCore-SageMaker compatible deployment file with streaming endpoint

import os
from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager
from bedrock_agentcore import BedrockAgentCoreApp
import mlflow

# MLflow configuration - tracking URI passed via environment variable
_MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', '')
_SAGEMAKER_MLFLOW_EXPERIMENT_NAME = "medical-triage-agentcore"
_REGION = os.environ.get('AWS_REGION', 'us-east-1')
_BEDROCK_MODELID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Set the SageMaker AI MLflow App Tracking URI
if _MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(_MLFLOW_TRACKING_URI)
    mlflow.autolog()

_mlflow_experiment_set = False

# Patient records database
patient_records = [
    {'id': 'PT-001', 'symptoms': 'chest pain, shortness of breath, sweating', 'age': 62, 'severity': 'high'},
    {'id': 'PT-002', 'symptoms': 'fever, cough, body aches', 'age': 35, 'severity': 'medium'},
    {'id': 'PT-003', 'symptoms': 'severe headache, stiff neck, sensitivity to light', 'age': 28, 'severity': 'high'},
    {'id': 'PT-004', 'symptoms': 'abdominal pain, nausea, vomiting', 'age': 45, 'severity': 'medium'},
    {'id': 'PT-005', 'symptoms': 'wheezing, difficulty breathing, coughing', 'age': 8, 'severity': 'high'},
]

# Treatment protocols
treatment_protocols = {
    'chest pain, shortness of breath, sweating': {'condition': 'Suspected Acute Myocardial Infarction', 'triage_level': 'Emergency', 'protocol': ['Immediate ECG', 'Administer aspirin', 'Prepare for cardiac catheterization']},
    'fever, cough, body aches': {'condition': 'Influenza-like Illness', 'triage_level': 'Urgent', 'protocol': ['Rapid flu test', 'Antipyretics', 'Rest and hydration']},
    'severe headache, stiff neck, sensitivity to light': {'condition': 'Suspected Meningitis', 'triage_level': 'Emergency', 'protocol': ['Lumbar puncture', 'IV antibiotics', 'CT scan']},
    'abdominal pain, nausea, vomiting': {'condition': 'Acute Gastroenteritis', 'triage_level': 'Urgent', 'protocol': ['IV fluids', 'Anti-emetics', 'Abdominal examination']},
    'wheezing, difficulty breathing, coughing': {'condition': 'Acute Asthma Exacerbation', 'triage_level': 'Emergency', 'protocol': ['Nebulized bronchodilator', 'Oxygen therapy', 'Peak flow measurement']},
}

app = BedrockAgentCoreApp()

# Medical Triage Agent System Prompt
MEDICAL_TRIAGE_PROMPT = """You are an expert medical triage assistant.

You help healthcare staff assess patients and recommend treatment protocols.

You have access to two tools:
1. symptom_lookup: Use this to retrieve a patient's symptoms by their patient ID
2. treatment_lookup: Use this to find the treatment protocol for a given set of symptoms

When given a patient ID:
1. First use symptom_lookup to get the patient's symptoms
2. Then use treatment_lookup with the EXACT symptom string returned by symptom_lookup
3. Present the findings clearly: patient info, condition, triage level, and treatment steps

Always use the tools in sequence. Do not guess symptoms or treatments."""

conversation_manager = SummarizingConversationManager(
    summary_ratio=0.3,
    preserve_recent_messages=5,
)

bedrock_model = BedrockModel(
    model_id=_BEDROCK_MODELID,
    region_name=_REGION,
    temperature=0.0,
)


@tool
def symptom_lookup(patient_id: str) -> str:
    """Look up patient symptoms by patient ID.

    Args:
        patient_id: The patient identifier (e.g., PT-001)

    Returns:
        A string with the patient's symptoms, age, and severity level.
    """
    patient_id = patient_id.upper().strip()
    for patient in patient_records:
        if patient['id'] == patient_id:
            return (
                f"Patient {patient['id']}: "
                f"Symptoms: {patient['symptoms']}. "
                f"Age: {patient['age']}. "
                f"Initial severity: {patient['severity']}."
            )
    return f"No patient found with ID: {patient_id}."


@tool
def treatment_lookup(symptoms: str) -> str:
    """Retrieve treatment protocol based on patient symptoms.

    Args:
        symptoms: The patient's symptoms as a comma-separated string

    Returns:
        A string with the condition, triage level, and treatment steps.
    """
    symptoms_clean = symptoms.lower().strip()
    for key, protocol in treatment_protocols.items():
        if key.lower() == symptoms_clean:
            steps = '\n'.join(protocol['protocol'])
            return (
                f"Condition: {protocol['condition']}\n"
                f"Triage Level: {protocol['triage_level']}\n"
                f"Treatment Protocol:\n{steps}"
            )
    return f"No matching treatment protocol found for symptoms: {symptoms}."


# Create the Medical Triage Agent
medical_triage_agent = Agent(
    model=bedrock_model,
    system_prompt=MEDICAL_TRIAGE_PROMPT,
    tools=[symptom_lookup, treatment_lookup],
    callback_handler=None,
)

@app.entrypoint
async def invoke(payload):
    """Medical triage agent function"""
    global _mlflow_experiment_set
    if not _mlflow_experiment_set and _MLFLOW_TRACKING_URI:
        try:
            mlflow.set_experiment(_SAGEMAKER_MLFLOW_EXPERIMENT_NAME)
            _mlflow_experiment_set = True
        except Exception as e:
            print(f"MLflow experiment setup deferred: {e}")
    print(payload)
    user_message = payload["prompt"]
    async for event in medical_triage_agent.stream_async(user_message):
        print(event)
        if "data" in event:
            print(event["data"])
            yield event["data"]


if __name__ == "__main__":
    app.run()

In [ ]:
# Configure AgentCore Runtime deployment settings
agentcore_runtime = Runtime()

agent_name = "medical_triage_agent"

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="strands_agentcore_SageMaker_deploy.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements-agentcore.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration=auth_config,
    disable_otel=True, # Required to capture traces in SageMaker AI MLflow
)

print("Configuration completed ✓")

### Launching agent to AgentCore Runtime

Now that we have a Dockerfile, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

In [ ]:
# Deploy agent to AgentCore Runtime (creates ECR repo and runtime)
print("Launching Agent server to AgentCore Runtime...")
print("This may take several minutes...")

launch_result = agentcore_runtime.launch(
    env_vars={
        "OTEL_PYTHON_EXCLUDED_URLS": "/ping,/invocations",
        "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
    }
)

print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

### Add SageMaker AI MLflow IAM permissions to the AgentCore agent IAM role 
We add sagemaker-mlflow IAM permission to the AgentCore runtime IAM role, to allow the AgentCore runtime to interact with SageMaker AI MLflow and log traces. 

See AWS documentation for IAM actions supported for [SageMaker AI MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-create-tracking-server-iam.html#mlflow-create-tracking-server-iam-actions)


In [ ]:
# Retrieve the IAM Role attached to the AgentCore runtime
agent_status = agentcore_runtime.status()
print(agent_status)
execution_role_arn = agent_status.config.execution_role
print(f"Agent execution role arn: {execution_role_arn}")

In [ ]:
# Import IAM permissions module
import boto3
import json
from utils.add_iam_permissions import add_sagemaker_mlflow_s3_permissions

In [ ]:
# update IAM role permissions on the AgentCore runtime
print("\nAdding SageMaker MLflow")
add_sagemaker_mlflow_s3_permissions(execution_role_arn)
print("✓ IAM permissions updated successfully")

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

In [ ]:
# Authenticate user and get bearer token for API access
bearer_token = reauthenticate_user(client_id=cognito_config["client_id"])

In [ ]:
def invoke_endpoint(
    agent_arn: str,
    payload,
    session_id: str,
    bearer_token: Optional[str],
    region: str = "us-east-1",
    endpoint_name: str = "DEFAULT",
) -> Any:
    """Invoke agent endpoint using HTTP request with bearer token."""
    escaped_arn = urllib.parse.quote(agent_arn, safe="")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    try:
        body = json.loads(payload) if isinstance(payload, str) else payload
    except json.JSONDecodeError:
        body = {"payload": payload}

    try:
        response = requests.post(
            url,
            params={"qualifier": endpoint_name},
            headers=headers,
            json=body,
            timeout=100,
            stream=True,
        )
        last_data = False
        for line in response.iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    last_data = True
                    line = line[6:]
                    line = line.replace('"', "")
                    yield line
                elif line:
                    line = line.replace('"', "")
                    if last_data:
                        yield "\n" + line
                    last_data = False

    except requests.exceptions.RequestException as e:
        print("Failed to invoke agent endpoint: %s", str(e))
        raise

In [ ]:
for chunk in invoke_endpoint(
    agent_arn=launch_result.agent_arn,
    payload={
        "prompt": "Can you help me triage patient PT-001? Look up their symptoms and provide the treatment protocol."
    },
    session_id=str(uuid.uuid4()),
    bearer_token=bearer_token,
):
    print(chunk.replace("\\n", "\n"), end="")

## Outputs in SageMaker AI MLflow
See the AgentCore runtime traced in the SageMaker AI MLflow App. 

- Go to the SageMaker AI MLflow App and open the `traces` tab for the MLflow experiment.
- If you used the default values in this notebook, your MLflow experiment will be `medical-triage-agentcore`
- Analyze the mlflow traces by clicking through various options: `Trace breakdown`, `Show execution timeline`, `Inputs / Outputs`, `Attributes`, `Events`. 
- Notice and track the usage details of your agent: `gen_ai.usage.total_tokens`, `system_prompt`, `gen_ai.request.model`. 

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime resources created.

In [ ]:
# Optional: Clean up Cognito user pool (commented out)
# delete_cognito_user_pool()

In [ ]:
# Get deployment details for cleanup (commented out)
# launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
# Optional: Delete AgentCore Runtime and ECR repository (commented out)
# agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
# ecr_client = boto3.client("ecr", region_name=region)

# runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,
# )

# response = ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split("/")[1], force=True
# )